# Quick stats for json results

In [5]:
from pathlib import Path
import pandas as pd

root_dir = Path.cwd().parent
data_dir = root_dir / "data"
results_dir = data_dir / "olmo7b_results"
dpo_results = results_dir / "olmo7b_dpo_given_all_temps.json"


In [9]:
df_dpo = pd.read_json(dpo_results)
df_dpo.describe()

,profile_id,temperature,response_number,mean_entropy,max_entropy,min_entropy,std_entropy
count,4400.00000,4400.000000,4400.000000,4400.000000,4400.000000,4.400000e+03,4400.000000
mean,374.50000,0.720000,3.000000,2.064488,4.820384,2.673129e-02,1.141243
std,278.60167,0.354441,1.414374,2.725055,3.310555,2.027784e-01,0.914209
min,1.00000,0.200000,1.000000,0.088352,0.957031,0.000000e+00,0.039062
25%,143.75000,0.500000,2.000000,0.454073,2.265625,6.571013e-36,0.521238
50%,374.50000,0.700000,3.000000,0.767699,3.234375,5.950000e-09,0.733066
75%,605.25000,1.000000,4.000000,1.914530,7.156250,4.392860e-05,1.491358
max,748.00000,1.200000,5.000000,9.933622,11.000000,5.156250e+00,4.080863


In [12]:
df_dpo["response"].head()

0    - Personality Traits: disciplined, resilient, ...
1    - Personality Traits: disciplined, loyal, resi...
2    - Personality Traits: disciplined, resilient, ...
3    - Personality Traits: disciplined, resilient, ...
4    - Personality Traits: disciplined, resilient, ...
Name: response, dtype: object

What about the sfts?

In [8]:
sft_file = results_dir / "olmo7b_sft_given_all_temps.json"
sft_df = pd.read_json(sft_file)

sft_df["temperature"].value_counts()

temperature
0.2    880
0.5    880
0.7    880
1.0    880
1.2    880
Name: count, dtype: int64

In [13]:
sft_df["response"].head()

0    -### Personal Information:\n- Name: John Doe\n...
1    -### Personal Information:\n- Name: John Doe\n...
2    -### Personal Information:\n- Name: John Doe\n...
3    -### Personal Information:\n- Name: John Doe\n...
4    -### Personal Information:\n- Name: John Doe\n...
Name: response, dtype: object

# Figuring out torch cumsum

In [56]:
import torch

rand = torch.tensor([0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.5,1.9,3, 3, 3.5])
rand.min(), rand.max()

(tensor(1.0000e-05), tensor(3.5000))

In [57]:
prob_dist = torch.softmax(rand, dim=-1)
prob_dist.min(), prob_dist.max(), prob_dist.sum(), len(prob_dist)

(tensor(0.0110), tensor(0.3654), tensor(1.0000), 14)

In [53]:
prob_dist

tensor([0.0208, 0.0208, 0.0208, 0.0208, 0.0208, 0.0208, 0.0208, 0.0208, 0.0208,
        0.0343, 0.0512, 0.0933, 0.0764, 0.1392, 0.4182])

In [60]:
top_p = 0.7

sorted_probs, sorted_indices = torch.sort(prob_dist, descending=True, dim=-1)
cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

# Create mask to remove tokens that are not in the top_p percentile
sorted_indices_to_remove = cumulative_probs > top_p
# print(sorted_indices_to_remove.shape)
sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
print(sorted_indices_to_remove[..., 1:], sorted_indices_to_remove[..., :-1])
sorted_indices_to_remove[..., 0] = False  # Ensure the most probable token is always kept
print(sorted_indices_to_remove)


# Map the mask back to original (unsorted) indices
indices_to_remove = torch.zeros_like(prob_dist, dtype=torch.bool)
indices_to_remove.scatter_(
    dim=-1,
    index=sorted_indices,
    src=sorted_indices_to_remove
)

# apply mask and renormalise
probs_nucleus = prob_dist.clone()
probs_nucleus[indices_to_remove] = 0.0
print(probs_nucleus)
probs_nucleus = probs_nucleus / probs_nucleus.sum(dim=-1, keepdim=True)


tensor([False, False,  True,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True]) tensor([False, False, False,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True])
tensor([False, False, False,  True,  True,  True,  True,  True,  True,  True,
         True,  True,  True,  True])
tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.2216, 0.2216, 0.3654])


In [59]:
probs_nucleus

tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.2741, 0.2741, 0.4519])